In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv("inventory_master.csv")

print(df.columns.tolist())
print(df.head())

['item_code', 'item_name', 'unit_price', 'annual_usage_qty', 'jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec', 'transactions', 'last_movement_date']
  item_code          item_name  unit_price  annual_usage_qty  jan  feb  mar  \
0    SKU068      Safety Helmet     4858.61              4297  442  183  459   
1    SKU003               Mask     4999.63              4016  262  354  471   
2    SKU040    Circuit Breaker     4636.85              3746  481  372   41   
3    SKU023  Cleaning Chemical     4723.85              3554  180  390  409   
4    SKU049     Hydraulic Hose     4841.12              3376  105  269  280   

   apr  may  jun  jul  aug  sep  oct  nov  dec  transactions  \
0  217  348  460  229  490  449  375  336  309            64   
1  448  146  375  279  413  383  379  141  365            69   
2  361  428  471   85  237  455  165  237  413            69   
3  370  340  352  157  272  311  303  468    2           230   
4  450   93  489  219 

In [3]:
# Create Inventory Value

df["inventory_value"] = (
    df["annual_usage_qty"]
    * df["unit_price"]
)

In [4]:
# Total Inventory Value

total_inventory_value = df["inventory_value"].sum()

print(total_inventory_value)

799300379.9499999


In [5]:
# SKU Ranking

inventory_ranking = (
    df.sort_values(
        "inventory_value",
        ascending=False
    )
)

inventory_ranking.head(10)

,item_code,item_name,unit_price,annual_usage_qty,jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec,transactions,last_movement_date,inventory_value
0,SKU068,Safety Helmet,4858.61,4297,442,183,459,217,348,460,229,490,449,375,336,309,64,2024-06-17,20877447.17
1,SKU003,Mask,4999.63,4016,262,354,471,448,146,375,279,413,383,379,141,365,69,2025-06-20,20078514.08
2,SKU040,Circuit Breaker,4636.85,3746,481,372,41,361,428,471,85,237,455,165,237,413,69,2026-01-19,17369640.10
3,SKU023,Cleaning Chemical,4723.85,3554,180,390,409,370,340,352,157,272,311,303,468,2,230,2026-02-05,16788562.90
4,SKU049,Hydraulic Hose,4841.12,3376,105,269,280,450,93,489,219,240,260,419,293,259,69,2024-09-29,16343621.12
5,SKU030,Fuse,4325.14,3747,160,302,446,307,203,28,399,180,333,405,485,499,0,2025-04-17,16206299.58
6,SKU010,Welding Rod,3804.57,4035,479,415,486,321,296,261,214,423,273,201,184,482,126,2024-10-13,15351439.95
7,SKU063,Hex Bolt,4481.31,3350,178,487,56,300,451,205,67,449,65,255,426,411,245,2025-01-30,15012388.50
8,SKU067,Pneumatic Cylinder,4775.91,2998,154,493,251,73,191,7,197,181,488,300,388,275,242,2025-03-19,14318178.18
9,SKU048,V Belt,4859.26,2906,181,405,442,461,301,310,121,83,184,296,11,111,77,2024-09-18,14121009.56


In [6]:
# Cumulative Percentage

inventory_ranking["cum_value"] = (
    inventory_ranking["inventory_value"]
    .cumsum()
)

inventory_ranking["cum_pct"] = (
    inventory_ranking["cum_value"]
    / inventory_ranking["inventory_value"].sum()
) * 100

In [9]:
# ABC Classification

inventory_ranking["ABC_Class"] = np.where(
    inventory_ranking["cum_pct"] <= 80,
    "A",
    np.where(
        inventory_ranking["cum_pct"] <= 95,
        "B",
        "C"
    )
)

In [10]:
# Class Summary

abc_summary = (
    inventory_ranking.groupby("ABC_Class")
    .agg(
        SKU_Count=("ABC_Class","count"),
        Inventory_Value=("inventory_value","sum")
    )
)

abc_summary

,SKU_Count,Inventory_Value
ABC_Class,,
A,56,6.340392e+08
B,22,1.248535e+08
C,22,4.040762e+07


In [12]:
# Top 10 SKUs

top_skus = (
    inventory_ranking[
        ["item_code","inventory_value","ABC_Class"]
    ]
    .head(10)
)

top_skus


,item_code,inventory_value,ABC_Class
0,SKU068,20877447.17,A
1,SKU003,20078514.08,A
2,SKU040,17369640.10,A
3,SKU023,16788562.90,A
4,SKU049,16343621.12,A
5,SKU030,16206299.58,A
6,SKU010,15351439.95,A
7,SKU063,15012388.50,A
8,SKU067,14318178.18,A
9,SKU048,14121009.56,A


In [13]:
# Export for Power BI

inventory_ranking.to_csv(
    "inventory_abc_analysis.csv",
    index=False
)

abc_summary.to_csv(
    "abc_summary.csv"
)